## HW2: Реализуйте модель для классификации изображений датасета FashionMNIST на PyTorch Lightning

Гандзюк Д.А. AITH25

In [ ]:
import os
import random
from dataclasses import dataclass

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import transforms

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

from torchmetrics.classification import F1Score, AUROC


@dataclass
class Config:
    seed: int = 42
    data_dir: str = "./data"
    batch_size: int = 128
    num_workers: int = 0  # у меня винда

    max_epochs: int = 15
    lr: float = 1e-3
    weight_decay: float = 1e-4

    early_stopping_patience: int = 3


cfg = Config()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


pl.seed_everything(cfg.seed, workers=True)
set_seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("pytorch_lightning:", pl.__version__)
print("cuda available:", torch.cuda.is_available())



Seed set to 42


torch: 2.7.0+cu118
torchvision: 0.22.0+cu118
pytorch_lightning: 2.6.0
cuda available: True


## 1) DataModule: загрузка, предобработка и DataLoader'ы

**Предобработка**:
- `ToTensor()` переводит изображения в тензор `float32` в диапазоне [0, 1]
- `Normalize(mean, std)` ускоряет и стабилизирует обучение

Для FashionMNIST:
- mean = 0.2860
- std  = 0.3530

**Разбиение**:
- `train` / `val` берём из оригинального train (60k) 55k/5k
- `test` — стандартная тестовая часть (10k)




In [2]:
class FashionMNISTDataModule(pl.LightningDataModule):
    def __init__(
        self,
        data_dir: str = "./data",
        batch_size: int = 128,
        num_workers: int = 0,
        seed: int = 42,
        train_size: int = 55_000,
        val_size: int = 5_000,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize(mean=(0.2860,), std=(0.3530,)),
            ]
        )

        self.train_ds = None
        self.val_ds = None
        self.test_ds = None

    def prepare_data(self) -> None:
        # скачивание (вызывается 1 раз)
        torchvision.datasets.FashionMNIST(self.hparams.data_dir, train=True, download=True)
        torchvision.datasets.FashionMNIST(self.hparams.data_dir, train=False, download=True)

    def setup(self, stage: str | None = None) -> None:
        if stage in (None, "fit"):
            full_train = torchvision.datasets.FashionMNIST(
                self.hparams.data_dir,
                train=True,
                transform=self.transform,
                download=False,
            )

            if self.hparams.train_size + self.hparams.val_size != len(full_train):
                raise ValueError(
                    f"train_size+val_size должно быть {len(full_train)}, "
                    f"получили {self.hparams.train_size + self.hparams.val_size}"
                )

            g = torch.Generator().manual_seed(int(self.hparams.seed))
            self.train_ds, self.val_ds = random_split(
                full_train,
                [self.hparams.train_size, self.hparams.val_size],
                generator=g,
            )

        if stage in (None, "test"):
            self.test_ds = torchvision.datasets.FashionMNIST(
                self.hparams.data_dir,
                train=False,
                transform=self.transform,
                download=False,
            )

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_ds,
            batch_size=self.hparams.batch_size,
            shuffle=True,
            num_workers=self.hparams.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_ds,
            batch_size=self.hparams.batch_size,
            shuffle=False,
            num_workers=self.hparams.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            self.test_ds,
            batch_size=self.hparams.batch_size,
            shuffle=False,
            num_workers=self.hparams.num_workers,
            pin_memory=torch.cuda.is_available(),
        )


dm = FashionMNISTDataModule(
    data_dir=cfg.data_dir,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    seed=cfg.seed,
)
dm.prepare_data()
dm.setup("fit")

xb, yb = next(iter(dm.train_dataloader()))
print(xb.shape, yb.shape, xb.min().item(), xb.max().item())



torch.Size([128, 1, 28, 28]) torch.Size([128]) -0.8101983666419983 2.022662878036499


## 2) LightningModule: модель + шаги обучения/валидации/теста

Выберем под FashionMNIST простой CNN, так как изображения 28×28, локальные паттерны важны; свёрточные слои обычно дают заметно лучше, чем MLP, при сопоставимом числе параметров (еще на CV смотрел лекцию по CNN)


**Метрики:**
- `F1Score(task="multiclass", num_classes=10, average="macro")` - одинаковый вклад каждого класса
- `AUROC(task="multiclass", num_classes=10, average="macro")` - ROC AUC для мультикласса




In [ ]:
class FashionMNISTModel(pl.LightningModule):
    def __init__(self, lr: float = 1e-3, weight_decay: float = 1e-4):
        super().__init__()
        self.save_hyperparameters()

        # Вход: [B, 1, 28, 28]
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 7x7
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10),
        )

        self.criterion = nn.CrossEntropyLoss()

        # Метрики
        self.val_f1 = F1Score(task="multiclass", num_classes=10, average="macro")
        self.val_auc = AUROC(task="multiclass", num_classes=10, average="macro")

        self.test_f1 = F1Score(task="multiclass", num_classes=10, average="macro")
        self.test_auc = AUROC(task="multiclass", num_classes=10, average="macro")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        logits = self.head(x)
        return logits

    def _shared_step(self, batch):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        probs = torch.softmax(logits, dim=1)
        return loss, probs, y

    def training_step(self, batch, batch_idx):
        loss, _, _ = self._shared_step(batch)
        self.log("train/loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, probs, y = self._shared_step(batch)

        self.val_f1.update(probs, y)
        self.val_auc.update(probs, y)

        self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/f1", self.val_f1, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/auc", self.val_auc, on_step=False, on_epoch=True, prog_bar=False)

    def test_step(self, batch, batch_idx):
        loss, probs, y = self._shared_step(batch)

        self.test_f1.update(probs, y)
        self.test_auc.update(probs, y)

        self.log("test/loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("test/f1", self.test_f1, on_step=False, on_epoch=True, prog_bar=True)
        self.log("test/auc", self.test_auc, on_step=False, on_epoch=True, prog_bar=False)

    def configure_optimizers(self):
        # AdamW хорошо работает для CNN и даёт аккуратную L2-регуляризацию через weight_decay
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )

        # CosineAnnealingLR: плавно уменьшает lr, часто даёт стабильную сходимость
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=cfg.max_epochs,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
            },
        }


model = FashionMNISTModel(lr=cfg.lr, weight_decay=cfg.weight_decay)
model



FashionMNISTModel(
  (backbone): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (head): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
  (criterion): CrossEntropyLoss()
  (val_f1): MulticlassF1Score()
  (val_auc): MulticlassAUROC()
  (test_f1): MulticlassF1Score()
  (test_auc): MulticlassAUROC()
)

## 3) Обучение через Trainer + TensorBoard

- `EarlyStopping(monitor="val/loss")` - останавливаемся, если качество на валидации перестало улучшаться
- `ModelCheckpoint(monitor="val/loss")` - сохраняем лучшую модель

**TensorBoard:**
- логи сохраняются в папку `tb_logs/`
- запуск из терминала:

```bash
tensorboard --logdir tb_logs
```



In [ ]:
# DataModule + модель
pl.seed_everything(cfg.seed, workers=True)

dm = FashionMNISTDataModule(
    data_dir=cfg.data_dir,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    seed=cfg.seed,
)

model = FashionMNISTModel(lr=cfg.lr, weight_decay=cfg.weight_decay)

# Логирование в TensorBoard
logger = TensorBoardLogger(save_dir="tb_logs", name="fashion_mnist")

# Callbacks
early_stopping = EarlyStopping(
    monitor="val/loss",
    mode="min",
    patience=cfg.early_stopping_patience,
)
checkpoint = ModelCheckpoint(
    monitor="val/loss",
    mode="min",
    save_top_k=1,
    filename="best-{epoch:02d}-{val_loss:.4f}",
)

trainer = pl.Trainer(
    max_epochs=cfg.max_epochs,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    logger=logger,
    callbacks=[early_stopping, checkpoint],
    log_every_n_steps=50,
)

trainer.fit(model, datamodule=dm)

print("Best checkpoint:", checkpoint.best_model_path)

trainer.test(model, datamodule=dm, ckpt_path="best")



Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 4070 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type              | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | backbone  | Sequential        | 19.0 K | train | 0    
1 | head      | Sequential        | 402 K  | train | 0    
2 | criterion | CrossEntropyLoss  | 0      | train | 0    
3 | val_f1    | MulticlassF1Score | 0      | train | 0    
4 | val_auc   | MulticlassAUROC   | 0      | train | 0    
5 | test_f1   | MulticlassF1Score | 0      | train | 0    
6 | test_auc  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

C:\Users\crazy\AppData\Roaming\Python\Python313\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
C:\Users\crazy\AppData\Roaming\Python\Python313\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Best checkpoint: tb_logs\fashion_mnist\version_0\checkpoints\best-epoch=05-val_loss=0.0000.ckpt


C:\Users\crazy\AppData\Roaming\Python\Python313\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test/auc            0.9951907396316528
         test/f1            0.9167203903198242
        test/loss           0.22715263068675995
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test/loss': 0.22715263068675995,
  'test/f1': 0.9167203903198242,
  'test/auc': 0.9951907396316528}]

## 4) Интерпретация TensorBoard и выводы


- **Train loss (`train/loss_epoch`, `train/loss_step`) убывает**:
  - `train/loss_epoch` монотонно снижается примерно до **~0.15** к концу обучения.
  - `train/loss_step` шумный, но общий тренд тоже вниз - примерно до **~0.10**.
  - Вывод: оптимизация идёт стабильно, модель продолжает учиться.

- **Validation loss (`val/loss`) тоже убывает**:
  - `val/loss` снижается примерно до **~0.225**.
  - Есть небольшие колебания.
  - **val loss не растёт**, значит явного переобучения на показанном участке не видно.

- **Метрики качества на валидации растут**:
  - `val/auc` поднимается примерно до **~0.995-0.996** значит модель очень хорошо разделяет классы
  - `val/f1` поднимается примерно до **~0.92** это хорошая классификация по всем классам в среднем (macro F1 даёт равный вес каждому классу).
  - Вывод: рост метрик согласован со снижением `val/loss`, значит улучшения правдивые.


### Почему выбраны optimizer и scheduler

- **AdamW**: устойчивый оптимизатор для CNN + корректная L2-регуляризация через `weight_decay`.
- **CosineAnnealingLR**: плавно уменьшает lr, часто даёт более стабильную сходимость и лучшее финальное качество.

